# 🔍 CholecSeg8k Label Analysis - With Directory Exploration

This version first explores the directory structure to find the correct paths.

In [1]:
# Cell 1: Imports
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict, Counter
from tqdm import tqdm
import pandas as pd
import json
import os

print("✅ Imports complete")

✅ Imports complete


In [2]:
# Cell 2: Explore Directory Structure
print("Exploring current directory structure...\n")

# Check current directory
current_dir = Path.cwd()
print(f"Current directory: {current_dir}")
print(f"\nContents of current directory:")
for item in sorted(current_dir.iterdir())[:20]:
    item_type = "DIR" if item.is_dir() else "FILE"
    print(f"  [{item_type}] {item.name}")

# Look for potential dataset directories
potential_dirs = [
    Path("cholecseg8k_raw"),
    Path("data/cholecseg8k_raw"),
    Path("../cholecseg8k_raw"),
    Path("CholecSeg8k"),
    Path("data/CholecSeg8k"),
    current_dir / "cholecseg8k_raw",
]

print("\nSearching for CholecSeg8k dataset...")
CHOLECSEG_DIR = None
for path in potential_dirs:
    if path.exists():
        print(f"  ✓ Found: {path}")
        CHOLECSEG_DIR = path
        break
    else:
        print(f"  ✗ Not found: {path}")

if CHOLECSEG_DIR is None:
    print("\n❌ ERROR: CholecSeg8k directory not found!")
    print("\nPlease update the path in the next cell.")
else:
    print(f"\n✅ Using: {CHOLECSEG_DIR}")

Exploring current directory structure...

Current directory: /raid/bsmse6/data

Contents of current directory:
  [DIR] .ipynb_checkpoints
  [FILE] 02_cholecseg8k_label_analysis__1.ipynb
  [FILE] 02_cholecseg8k_label_analysis_debug.ipynb
  [FILE] 02_extract_blood_labels.ipynb
  [FILE] 03_train_unet_blood.ipynb
  [FILE] 1_Dataset Exploration.ipynb
  [FILE] 2_Segmentation Dataset.ipynb
  [FILE] 3_Extract the blood frames.ipynb
  [FILE] 4_Training U-Net++ Model.ipynb
  [FILE] 5_U-Net++ on Cholec80.ipynb
  [FILE] NOTEBOOK_2_CORRECTIONS_SUMMARY.md
  [FILE] README_BLOOD_SEGMENTATION.md
  [FILE] UNETPP_ARCHITECTURE_CODE.md
  [DIR] cholec80
  [DIR] cholecseg8k_blood
  [DIR] cholecseg8k_raw
  [DIR] label_analysis
  [FILE] mask_analysis_data.csv
  [FILE] mask_quality_analysis.png
  [FILE] medsam_results_overview.png

Searching for CholecSeg8k dataset...
  ✓ Found: cholecseg8k_raw

✅ Using: cholecseg8k_raw


In [3]:
# Cell 3: Manual Path Override (if needed)
# If the automatic detection failed, manually set the path here:

# CHOLECSEG_DIR = Path("/path/to/your/cholecseg8k_raw")  # Uncomment and update

print(f"Dataset directory: {CHOLECSEG_DIR}")
print(f"Exists: {CHOLECSEG_DIR.exists()}")

Dataset directory: cholecseg8k_raw
Exists: True


In [4]:
# Cell 4: Explore Dataset Structure
if CHOLECSEG_DIR and CHOLECSEG_DIR.exists():
    print("\nExploring dataset structure...\n")
    
    # List all items in dataset directory
    items = sorted(list(CHOLECSEG_DIR.iterdir()))
    print(f"Found {len(items)} items in {CHOLECSEG_DIR.name}:")
    for item in items[:10]:
        item_type = "DIR" if item.is_dir() else "FILE"
        print(f"  [{item_type}] {item.name}")
    if len(items) > 10:
        print(f"  ... and {len(items) - 10} more")
    
    # Check for video directories
    video_dirs = [d for d in items if d.is_dir() and 'video' in d.name.lower()]
    print(f"\nFound {len(video_dirs)} video directories")
    
    if len(video_dirs) > 0:
        # Explore first video directory
        first_video = video_dirs[0]
        print(f"\nExploring {first_video.name}:")
        subdirs = list(first_video.iterdir())
        for subdir in subdirs:
            subdir_type = "DIR" if subdir.is_dir() else "FILE"
            print(f"  [{subdir_type}] {subdir.name}")
            
            # If it's a directory, show a few files
            if subdir.is_dir():
                files = list(subdir.glob("*.*"))[:3]
                for f in files:
                    print(f"      - {f.name}")
                if len(list(subdir.glob("*.*"))) > 3:
                    print(f"      ... ({len(list(subdir.glob('*.*')))} files total)")
    else:
        print("\n⚠️  No 'video' directories found. Please check structure.")
else:
    print("❌ Dataset directory not found. Please set CHOLECSEG_DIR correctly.")


Exploring dataset structure...

Found 17 items in cholecseg8k_raw:
  [DIR] video01
  [DIR] video09
  [DIR] video12
  [DIR] video17
  [DIR] video18
  [DIR] video20
  [DIR] video24
  [DIR] video25
  [DIR] video26
  [DIR] video27
  ... and 7 more

Found 17 video directories

Exploring video01:
  [DIR] video01_16345
      - frame_16405_endo_watershed_mask.png
      - frame_16385_endo_mask.png
      - frame_16351_endo_watershed_mask.png
      ... (320 files total)
  [DIR] video01_28580
      - frame_28640_endo_mask.png
      - frame_28633_endo_mask.png
      - frame_28596_endo_mask.png
      ... (320 files total)
  [DIR] video01_28820
      - frame_28826_endo.png
      - frame_28842_endo_watershed_mask.png
      - frame_28845_endo_watershed_mask.png
      ... (320 files total)
  [DIR] video01_16425
      - frame_16483_endo_color_mask.png
      - frame_16447_endo_watershed_mask.png
      - frame_16460_endo_watershed_mask.png
      ... (320 files total)
  [DIR] video01_14859
      - frame_14

In [5]:
# Cell 5: Find Mask Directory Name
if CHOLECSEG_DIR and CHOLECSEG_DIR.exists():
    video_dirs = sorted([d for d in CHOLECSEG_DIR.iterdir() if d.is_dir()])
    
    if len(video_dirs) > 0:
        first_video = video_dirs[0]
        subdirs = [d for d in first_video.iterdir() if d.is_dir()]
        
        print("Searching for mask directory...\n")
        
        MASK_SUBDIR_NAME = None
        possible_names = ['watershed_mask', 'mask', 'watershed', 'annotation', 'label']
        
        for subdir in subdirs:
            subdir_lower = subdir.name.lower()
            if any(name in subdir_lower for name in possible_names):
                MASK_SUBDIR_NAME = subdir.name
                print(f"✓ Found mask directory: {MASK_SUBDIR_NAME}")
                
                # Check for PNG files
                png_files = list(subdir.glob("*.png"))
                print(f"  Contains {len(png_files)} PNG files")
                
                if len(png_files) > 0:
                    # Load and analyze first mask
                    sample_mask = cv2.imread(str(png_files[0]), cv2.IMREAD_GRAYSCALE)
                    if sample_mask is not None:
                        print(f"  Sample mask shape: {sample_mask.shape}")
                        print(f"  Sample mask dtype: {sample_mask.dtype}")
                        unique_vals = np.unique(sample_mask)
                        print(f"  Sample unique values: {unique_vals[:10]}..." if len(unique_vals) > 10 else f"  Sample unique values: {unique_vals}")
                break
        
        if MASK_SUBDIR_NAME is None:
            print("❌ Could not find mask directory!")
            print("\nAvailable subdirectories:")
            for subdir in subdirs:
                print(f"  - {subdir.name}")
else:
    print("❌ Dataset directory not set")

Searching for mask directory...

❌ Could not find mask directory!

Available subdirectories:
  - video01_16345
  - video01_28580
  - video01_28820
  - video01_16425
  - video01_14859
  - video01_00080
  - video01_15099
  - video01_00400
  - video01_28660
  - video01_00160
  - video01_14939
  - video01_00240
  - video01_15019
  - video01_28740
  - video01_28900
  - video01_16585


In [6]:
# Cell 6: Setup Output Directory
OUTPUT_DIR = Path("label_analysis")
OUTPUT_DIR.mkdir(exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

Output directory: label_analysis


In [7]:
# Cell 7: Scan All Masks (with correct paths)
if CHOLECSEG_DIR and CHOLECSEG_DIR.exists() and MASK_SUBDIR_NAME:
    print("Scanning all masks for unique labels...\n")
    
    all_labels = set()
    label_counts = Counter()
    video_label_map = defaultdict(set)
    
    video_dirs = sorted([d for d in CHOLECSEG_DIR.iterdir() if d.is_dir()])
    
    for video_dir in tqdm(video_dirs, desc="Scanning videos"):
        mask_dir = video_dir / MASK_SUBDIR_NAME
        
        if not mask_dir.exists():
            continue
        
        mask_files = list(mask_dir.glob("*.png"))
        if len(mask_files) == 0:
            continue
            
        sample_size = min(50, len(mask_files))
        sampled_masks = np.random.choice(mask_files, sample_size, replace=False)
        
        for mask_path in sampled_masks:
            mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
            if mask is None:
                continue
            
            unique_labels = np.unique(mask)
            all_labels.update(unique_labels)
            
            for label in unique_labels:
                label_counts[label] += np.sum(mask == label)
                video_label_map[label].add(video_dir.name)
    
    all_labels = sorted(all_labels)
    
    if len(all_labels) > 0:
        print(f"\n✅ Scan complete!")
        print(f"   Total unique labels found: {len(all_labels)}")
        print(f"   Label range: {min(all_labels)} to {max(all_labels)}")
        print(f"   All labels: {all_labels}")
    else:
        print("\n❌ No labels found! Check directory structure.")
else:
    print("❌ Cannot proceed - dataset or mask directory not found")

❌ Cannot proceed - dataset or mask directory not found


In [8]:
# Cell 8: Create Frequency Table
if len(all_labels) > 0:
    label_data = []
    for label in all_labels:
        label_data.append({
            'Label': label,
            'Total Pixels': label_counts[label],
            'Videos': len(video_label_map[label]),
            'Percentage': 100 * label_counts[label] / sum(label_counts.values())
        })
    
    df = pd.DataFrame(label_data)
    df = df.sort_values('Total Pixels', ascending=False)
    
    print("Top 20 Most Frequent Labels:")
    print(df.head(20).to_string(index=False))
    
    df.to_csv(OUTPUT_DIR / "label_frequency_table.csv", index=False)
    print(f"\n📊 Full table saved: {OUTPUT_DIR / 'label_frequency_table.csv'}")
else:
    print("No labels to analyze")

NameError: name 'all_labels' is not defined

In [9]:
# Cell 9: Known Class Mappings
KNOWN_CLASSES = {
    0: "Background",
    11: "Abdominal Wall",
    12: "Liver",
    13: "Gastrointestinal Tract",
    14: "Fat",
    15: "Grasping Retractor",
    21: "Grasper (Instrument)",
    22: "Bipolar (Instrument)",
    23: "Hook (Instrument)",
    24: "Blood",
    25: "Scissors (Instrument)",
    26: "Clipper (Instrument)",
    27: "Irrigator (Instrument)",
    28: "Specimen Bag",
    31: "Suction Instrument",
    32: "Hepatic Vein",
    33: "Liver Ligament",
}

INSTRUMENT_CLASSES = {
    21: "Grasper",
    22: "Bipolar",
    23: "Hook",
    25: "Scissors",
    26: "Clipper",
    27: "Irrigator",
    28: "Specimen Bag",
    31: "Suction"
}

if len(all_labels) > 0:
    print("Known Surgical Instruments:")
    for label in sorted(INSTRUMENT_CLASSES.keys()):
        status = "✓ FOUND" if label in all_labels else "✗ NOT FOUND"
        print(f"  Class {label:2d}: {INSTRUMENT_CLASSES[label]:20s} - {status}")
    
    print("\nBlood:")
    status = "✓ FOUND" if 24 in all_labels else "✗ NOT FOUND"
    print(f"  Class 24: Blood - {status}")
    
    print("\nUnknown Labels (not in documentation):")
    unknown = [l for l in all_labels if l not in KNOWN_CLASSES]
    if unknown:
        print(f"  Found {len(unknown)} unknown labels: {unknown}")
    else:
        print("  None - all labels documented!")

NameError: name 'all_labels' is not defined

In [10]:
# Cell 10: Save Class Mapping
if len(all_labels) > 0:
    INSTRUMENT_MAPPING = {
        class_id: class_name 
        for class_id, class_name in INSTRUMENT_CLASSES.items() 
        if class_id in all_labels
    }
    
    if 24 in all_labels:
        INSTRUMENT_MAPPING[24] = "Blood"
    
    print("Final Class Mapping (classes found in dataset):")
    for class_id, class_name in sorted(INSTRUMENT_MAPPING.items()):
        print(f"  {class_id}: {class_name}")
    
    # Save as JSON
    mapping_path = OUTPUT_DIR / "class_mapping.json"
    with open(mapping_path, 'w') as f:
        json.dump(INSTRUMENT_MAPPING, f, indent=2)
    
    print(f"\n💾 Saved: {mapping_path}")
    
    # Save as Python dict
    dict_path = OUTPUT_DIR / "class_mapping.py"
    with open(dict_path, 'w') as f:
        f.write("# CholecSeg8k Class Mapping - Auto-generated\n\n")
        f.write("CLASS_MAPPING = {\n")
        for class_id, class_name in sorted(INSTRUMENT_MAPPING.items()):
            f.write(f"    {class_id}: '{class_name}',\n")
        f.write("}\n\n")
        f.write("# Instrument classes only (excluding blood)\n")
        f.write("INSTRUMENT_CLASSES = {\n")
        for class_id, class_name in sorted(INSTRUMENT_MAPPING.items()):
            if class_id != 24:
                f.write(f"    {class_id}: '{class_name}',\n")
        f.write("}\n")
    
    print(f"💾 Saved: {dict_path}")
    
    print(f"\n✅ Analysis complete! Found {len(INSTRUMENT_MAPPING)} classes.")

NameError: name 'all_labels' is not defined